# Building a Research Assistant with Web + X Search

What do reputable news sources say about a topic? What does the crowd on X think? And where do those two stories diverge?

In this guide, we'll build a research assistant that cross-references reporting from vetted news outlets with public discourse on X, then produces a structured "divergence briefing" that clusters claims into five categories: consensus, X-ahead-of-press, press-ahead-of-X, X-only, and press-only.

This takes advantage of Grok's built-in web search and X search as native tools, with domain filtering to scope web results to trusted outlets and inline citations linking every claim back to its source.

### What we'll use
- Web search with domain filtering (`allowed_domains`) to restrict results to trusted outlets
- X search to capture real-time public discourse
- Inline citations linking every claim back to its source
- Structured output with Pydantic models to parse the final analysis
- The native xai-sdk

### Table of Contents
- [Setup](#setup)
- [Act 1: Web Search with Domain Filtering](#act-1-web-search-with-domain-filtering)
- [Interlude: Filtered vs. Unfiltered](#interlude-filtered-vs-unfiltered)
- [Act 2: X Search for Public Discourse](#act-2-x-search-for-public-discourse)
- [Act 3: The Divergence Analysis](#act-3-the-divergence-analysis)
- [Structured Output](#structured-output)
- [Putting It All Together](#putting-it-all-together)
- [Conclusion](#conclusion)

## Setup

We use the native xai-sdk, which gives us direct access to Grok's search tools as first-class citizens, including domain filtering, X handle filtering, and inline citations.

All you need is an xAI API key.

In [1]:
%%capture
%pip install -q xai-sdk python-dotenv

In [2]:
import os

from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import system, user
from xai_sdk.tools import web_search, x_search

load_dotenv()

client = Client(api_key=os.getenv("XAI_API_KEY"))

MODEL = "grok-4.20-reasoning"

## Act 1: Web Search with Domain Filtering

Grok's `web_search` tool lets you restrict results to specific domains using `allowed_domains` (max 5 per call). Instead of hoping the model finds good sources, you tell it where to look.

We'll define curated domain lists for different research contexts, then run a search scoped to major news outlets.

In [3]:
# Curated domain lists for different research contexts
NEWS_DOMAINS = ["reuters.com", "apnews.com", "bbc.com", "ft.com", "wsj.com"]
TECH_DOMAINS = ["techcrunch.com", "arstechnica.com", "theverge.com", "wired.com"]
SCIENCE_DOMAINS = ["nature.com", "sciencedirect.com", "newscientist.com"]

In [4]:
TOPIC = "Space-based data centres and the future of orbital computing infrastructure"

chat = client.chat.create(
    model=MODEL,
    tools=[web_search(allowed_domains=NEWS_DOMAINS)],
    include=["inline_citations"],
)
chat.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims from these sources, with citations. Be concise."
))

response = None
for response, chunk in chat.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key claims from recent reporting (primarily late 2025–early 2026):**

- **Space-based data centers offer major advantages for AI workloads**, including constant 24/7 solar power, the ability to radiate heat directly into the vacuum of space (eliminating water-intensive cooling), and bypassing Earth’s power-grid and land constraints amid surging AI demand. Musk has claimed space could become the lowest-cost location for AI computing “within two years, three at the latest.”[[1]](https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/)[[2]](https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/)

- **Ambitious deployment plans are underway from multiple players**: SpaceX is seeking approvals for up to 1 million solar-powered data-center satellites (with IPO funding), while Blue Origin (Project Sunrise), Starcloud (targeting an 88,000-satellite con

Every claim is backed by an inline citation linking to the original article. Let's inspect those citations programmatically:

In [5]:
print("Web search citations:")
for citation in response.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

Web search citations:
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.bbc.com/news/articles/cjewvpkw7weo
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/


Notice every source is from our `NEWS_DOMAINS` list.

## Interlude: Filtered vs. Unfiltered

What happens if we run the same query without domain filtering? The model is free to pull from any source. This isn't a controlled experiment (ranking, freshness, and source availability all vary between calls), but it illustrates why scoping your sources matters for research.

In [6]:
chat_unfiltered = client.chat.create(
    model=MODEL,
    tools=[web_search()],
    include=["inline_citations"],
)
chat_unfiltered.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims with citations. Be concise."
))

response_unfiltered = None
for response_unfiltered, chunk in chat_unfiltered.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key claims from recent 2025–2026 reporting:**

- **Prototypes are now in orbit and early AI demonstrations have succeeded.** Axiom Space launched the first dedicated orbital data center (ODC) nodes to low-Earth orbit on January 11, 2026, building on prior ISS tests with AWS and Red Hat tech for cloud computing, AI/ML, and secure processing.[[1]](https://www.axiomspace.com/orbital-data-center)[[1]](https://www.axiomspace.com/orbital-data-center) Starcloud (NVIDIA-backed) launched an H100 GPU in late 2025, trained an LLM in space, and ran Google Gemini, with further higher-power missions planned.[[2]](https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk)

- **Continuous solar power is the core advantage**, offering near-constant energy in sun-synchronous orbits (up to ~8x more productive than terrestrial solar) plus vacuum radiative cooling, addressing Earth’s power-grid and energy constraints for AI workloads with no land or water footprint.[[3]](http

In [7]:
print("\nUnfiltered citations:")
for citation in response_unfiltered.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

print(f"\nFiltered: {len(response.inline_citations)} citations (all from vetted outlets)")
print(f"Unfiltered: {len(response_unfiltered.inline_citations)} citations (mixed sources)")


Unfiltered citations:
  https://www.axiomspace.com/orbital-data-center
  https://www.axiomspace.com/orbital-data-center
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://www.scientificamerican.com/article/data-centers-in-space/
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://en.wikipedia.org/wiki/Space-based_data_center
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://arstechnica.com/space/2026/03/orbital-data-centers-part-1-theres-no-way-this-is-economically-viable-right/

Filtered: 6 citations (all from vetted outlets)
Unfiltered: 11 citations (mixed sources

Both produce useful summaries, but the filtered version gives you confidence in where the information came from.

## Act 2: X Search for Public Discourse

Now let's see what people are actually saying. Grok's `x_search` tool searches X directly, with optional handle filtering and date ranges.

In [8]:
chat_x = client.chat.create(
    model=MODEL,
    tools=[x_search()],
    include=["inline_citations"],
)
chat_x.append(user(
    f"Search X for what people are saying about: {TOPIC}. "
    "Summarize the 3-5 key themes in public sentiment, with citations to specific posts. Be concise."
))

response_x = None
for response_x, chunk in chat_x.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**Key themes in public sentiment on X regarding space-based data centers and orbital computing:**

**1. Strong emphasis on energy and cooling advantages.** Many highlight constant solar power in sun-synchronous orbits (no atmosphere or day/night cycles) and passive radiative cooling into space vacuum as major wins over terrestrial data centers' power grid strain and water use.[[1]](https://x.com/renaldbarnett/status/2040189083071471955)[[2]](https://x.com/kimmonismus/status/1980945551995863217)[[3]](https://x.com/Jackson_Metrics/status/2038842657112543594)[[4]](https://x.com/niccruzpatane/status/2020175251837985052)

**2. Optimism about economic viability from falling launch costs.** Discussions frequently note that Starship-driven cost reductions (potentially to <$200/kg) could make orbital compute competitive with Earth-based energy costs, especially for AI workloads.[[5]](https://x.com/rmcentush/status/1985787187556991364)[[6]](https://x.com/aaronburnett/status/2029686439546605591)


In [9]:
print("X search citations:")
for citation in response_x.inline_citations:
    if citation.HasField("x_citation"):
        print(f"  {citation.x_citation.url}")

X search citations:
  https://x.com/renaldbarnett/status/2040189083071471955
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/Jackson_Metrics/status/2038842657112543594
  https://x.com/niccruzpatane/status/2020175251837985052
  https://x.com/rmcentush/status/1985787187556991364
  https://x.com/aaronburnett/status/2029686439546605591
  https://x.com/Andercot/status/1981465914400002550
  https://x.com/venturemanny/status/2038684453946405096
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/venturemanny/status/2038684453946405096
  https://x.com/Andercot/status/1981465914400002550


Compare the tone and content with Act 1. Press reporting tends to balance the opportunity against economic and technical skepticism, often citing analyst doubts. X discussion leans more optimistic, focusing on the physics and engineering case for why orbital compute could work. The overlap and divergence between those perspectives is what we'll formalize next.

## Act 3: The Divergence Analysis

We take the full outputs from both searches and ask Grok to reconcile them, clustering every claim into one of five categories:

1. CONSENSUS: claims both sources agree on
2. X_AHEAD_OF_PRESS: claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X: claims in press but not discussed on X
4. X_ONLY: claims unique to X with no press corroboration
5. PRESS_ONLY: claims unique to press with no social discussion

The three-pass architecture:
- Pass 1 (already done): Web search with domain filtering
- Pass 2 (already done): X search
- Pass 3 (below): A reconciliation call with no tools, just analysis

A caveat: the temporal labels ("X-ahead-of-press", etc.) reflect the model's assessment at query time, not timestamped provenance. This is a pattern demo, not a rigorous fact-checker.

In [10]:
from pydantic import BaseModel


class Claim(BaseModel):
    text: str
    source: str
    category: str


class ClaimCluster(BaseModel):
    category: str
    claims: list[Claim]


class DivergenceBrief(BaseModel):
    topic: str
    clusters: list[ClaimCluster]
    executive_summary: str

In [11]:
RECONCILIATION_PROMPT = """You are a research analyst. Given web search findings from \
reputable sources and X/social media findings on the same topic, cluster the claims \
into exactly five categories:
1. CONSENSUS — claims both sources agree on
2. X_AHEAD_OF_PRESS — claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X — claims in press but not discussed on X
4. X_ONLY — claims unique to X with no press corroboration
5. PRESS_ONLY — claims unique to press with no social discussion

For each claim, note the source ("Press" or "X") and category. Be thorough: extract \
every distinct claim from both inputs. Output as JSON matching the provided schema."""

# Pass 1 and 2 outputs become the context for Pass 3
web_findings = response.content
x_findings = response_x.content

chat_reconcile = client.chat.create(
    model=MODEL,
    response_format=DivergenceBrief,
)
chat_reconcile.append(system(RECONCILIATION_PROMPT))
chat_reconcile.append(user(
    f"## Web Search Findings\n{web_findings}\n\n## X Search Findings\n{x_findings}"
))

response_reconcile = None
for response_reconcile, chunk in chat_reconcile.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

{
  "topic": "Space-Based Data Centers for AI Workloads",
  "clusters": [
    {
      "category": "CONSENSUS",
      "claims": [
        {
          "text": "Space-based data centers offer major advantages including constant 24/7 solar power, radiating heat directly into space vacuum (eliminating water-intensive cooling), and bypassing Earth’s power-grid and land constraints amid surging AI demand.",
          "source": "Press",
          "category": "CONSENSUS"
        },
        {
          "text": "Constant solar power in sun-synchronous orbits with no atmosphere or day/night cycles and passive radiative cooling into space vacuum are major advantages over terrestrial data centers' power grid strain and water use.",
          "source": "X",
          "category": "CONSENSUS"
        },
        {
          "text": "Significant technical, economic, and practical challenges including radiation hardening, difficult maintenance/upgrades in orbit, and issues with rapid scalability.",
      

## Structured Output

The reconciliation call used `response_format=DivergenceBrief` to get structured JSON. Let's parse it into our Pydantic model and display it cleanly.

In [12]:
brief = DivergenceBrief.model_validate_json(response_reconcile.content)

print(f"Topic: {brief.topic}")
print(f"{'=' * 60}")
for cluster in brief.clusters:
    print(f"\n{cluster.category} ({len(cluster.claims)} claims)")
    print("-" * 40)
    for claim in cluster.claims:
        print(f"  [{claim.source}] {claim.text}")

print(f"\n{'=' * 60}")
print(f"Executive Summary:\n{brief.executive_summary}")

Topic: Space-Based Data Centers for AI Workloads

CONSENSUS (6 claims)
----------------------------------------
  [Press] Space-based data centers offer major advantages including constant 24/7 solar power, radiating heat directly into space vacuum (eliminating water-intensive cooling), and bypassing Earth’s power-grid and land constraints amid surging AI demand.
  [X] Constant solar power in sun-synchronous orbits with no atmosphere or day/night cycles and passive radiative cooling into space vacuum are major advantages over terrestrial data centers' power grid strain and water use.
  [Press] Significant technical, economic, and practical challenges including radiation hardening, difficult maintenance/upgrades in orbit, and issues with rapid scalability.
  [X] Major technical and operational challenges such as difficulties in maintenance/replacement of hardware and radiation hardening.
  [Press] Ambitious deployment plans from multiple players including SpaceX and Starcloud targeting 